# Create Rockefeller Foundation Awards

Creates Rockefeller Foundation awards. ~1,511 grants. The script
combines the WP REST `/wp-json/wp/v2/grant` endpoint (for URL list
and taxonomy refs) with HTML scraping of each landing page (for
amount, term, focus area, description) — the WP REST ACF block is
empty.

**Prerequisites:**
- Run `scripts/local/rockefeller_to_s3.py` to download and upload the data.

**Data source:** rockefellerfoundation.org WP REST + landing pages
**S3 location:** `s3a://openalex-ingest/awards/rockefeller/rockefeller_projects.parquet`

**Rockefeller funder (OpenAlex):**
- funder_id: 4320306149
- ROR: https://ror.org/03sfkwk85
- DOI: 10.13039/100000877
- display_name: "Rockefeller Foundation"

**Schema notes:**
- Currency hardcoded USD by the download script.
- Amount is parsed by the script into `amount_usd` already; `amount_raw`
  preserved for traceability.
- Dates parsed from "MM.DD.YYYY - MM.DD.YYYY" pattern; year-only fallback
  used when needed.
- Focus area / regions from WP custom taxonomies.


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rockefeller_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/rockefeller/rockefeller_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.rockefeller_raw;

## Step 1.5: Inspect raw + verify amount/currency

In [ ]:
%sql
DESCRIBE openalex.awards.rockefeller_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.rockefeller_raw LIMIT 5;

In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(amount_usd) AS has_amount,
  ROUND(MIN(amount_usd), 0) AS min_amt,
  ROUND(MAX(amount_usd), 0) AS max_amt,
  ROUND(AVG(amount_usd), 0) AS avg_amt,
  collect_set(currency) AS currencies
FROM openalex.awards.rockefeller_raw;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rockefeller_awards
USING delta
AS
WITH rockefeller_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306149
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.slug)))) % 9000000000 AS id,
    g.title AS display_name,
    NULLIF(g.description_raw, '') AS description,
    f.funder_id,
    g.slug AS funder_award_id,
    TRY_CAST(g.amount_usd AS DOUBLE) AS amount,
    g.currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    NULLIF(g.focus_area_raw, '') AS funder_scheme,
    'rockefeller_wp' AS provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') AS end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) AS start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) AS end_year,
    -- Rockefeller funds organisations; grantee name is in the title (most pages just
    -- name the organisation followed by year, e.g. "Global Methane Hub Stichting 2026").
    struct(
        CAST(NULL AS STRING) AS given_name,
        CAST(NULL AS STRING) AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            COALESCE(NULLIF(g.grantee_raw, ''), g.title) AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    g.url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.slug)))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.rockefeller_raw g
CROSS JOIN rockefeller_funder f
WHERE g.slug IS NOT NULL AND TRIM(g.slug) != '';

## Step 3: Insert into openalex_awards_raw at priority 40

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rockefeller_wp' AND priority = 40;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    40 AS priority
FROM openalex.awards.rockefeller_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_rockefeller_awards FROM openalex.awards.rockefeller_awards;

In [ ]:
%sql
-- Step 6.7 fail-fast amount/currency check
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(amount), 0) AS avg_amt
FROM openalex.awards.rockefeller_awards;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       amount, currency, start_date, end_date, funder_scheme,
       lead_investigator.affiliation.name AS grantee
FROM openalex.awards.rockefeller_awards LIMIT 10;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) FROM openalex.awards.rockefeller_awards GROUP BY funder.display_name;